In [1]:
import numpy as np

In [2]:
import pandas as pd
df= pd.read_csv("../Datasets/heart_disease.csv", na_values= "?").dropna()

In [3]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [4]:
df["target"]= (df["num"]>0).astype(int)
df["target"]

0      0
1      1
2      1
3      0
4      0
      ..
297    1
298    1
299    1
300    1
301    1
Name: target, Length: 297, dtype: int32

In [5]:
#assingning values
X= df.drop(columns= ["num", "target"])
y= df["target"]

print("rows:", len(X), "features:", X.shape[1])

rows: 297 features: 13


### 1. The problem: one train_test_split is unreliable
we can see with 2 model SVM and Random Forest

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test= train_test_split(
    X, y, test_size=0.2, random_state=10
)

#### SVM
start with SVM

In [7]:
from sklearn.svm import SVC
svm= SVC()

svm.fit(X_train, y_train)
svm.score(X_test, y_test)  #model score with unseen data

0.6666666666666666

### Random Forest

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf= RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf.score(X_test, y_test)

0.8666666666666667

### KFold

In [9]:
from sklearn.model_selection import KFold

kf= KFold(n_splits=3)
kf

KFold(n_splits=3, random_state=None, shuffle=False)

In [10]:
data= [1, 2, 3, 4, 5, 6, 7, 8, 9]

for train_index, val_index in kf.split(data):
    
    print(train_index, val_index)#index values not actual data, every one gets chance

[3 4 5 6 7 8] [0 1 2]
[0 1 2 6 7 8] [3 4 5]
[0 1 2 3 4 5] [6 7 8]


In [13]:
#now we are supplying KFold to our variables and comparesing the scores
kf= KFold(n_splits=3)

svm_scores= []
rf_scores= []

#running a for loop for 3 time like we are specifyed above {kf= KFold(n_splits=3)}
for train_index, test_index in kf.split(X, y):
    X_train, X_test, y_train, y_test= X.iloc[train_index], X.iloc[test_index], y.iloc[train_index], y.iloc[test_index]

    svm= SVC()
    svm.fit(X_train, y_train)
    svm_scores.append(svm.score(X_test, y_test))
    

    rf= RandomForestClassifier(n_estimators=40, random_state=42)
    rf.fit(X_train, y_train)
    rf_scores.append(rf.score(X_test, y_test))
    
    

In [17]:

from sklearn.model_selection import KFold
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

svm_scores = []
rf_scores = []

kf = KFold(n_splits=3)

for train_index, test_index in kf.split(X, y):

    # Split features and their matching target labels
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]  # Fixed

    # SVM
    svm = SVC()
    svm.fit(X_train, y_train)
    svm_scores.append(svm.score(X_test, y_test))

    # Random Forest
    rf = RandomForestClassifier(
        n_estimators=40,
        random_state=42
    )
    rf.fit(X_train, y_train)
    rf_scores.append(rf.score(X_test, y_test))

print("SVM scores:", svm_scores)
print("SVM mean accuracy:", sum(svm_scores) / len(svm_scores))

print("Random Forest scores:", rf_scores)
print("Random Forest mean accuracy:", sum(rf_scores) / len(rf_scores))


SVM scores: [0.6161616161616161, 0.6565656565656566, 0.6060606060606061]
SVM mean accuracy: 0.6262626262626263
Random Forest scores: [0.8383838383838383, 0.8080808080808081, 0.797979797979798]
Random Forest mean accuracy: 0.8148148148148149


### cross_val_score --> is usually the simplest way to perform cross-validation without manually writing a for loop.

In [15]:
from sklearn.model_selection import cross_val_score

In [16]:
#for SVM
cross_val_score(svm, X, y, cv= kf)

array([0.61616162, 0.65656566, 0.60606061])

In [18]:
#for RandomForestClassifier
cross_val_score(rf, X, y, cv= kf)

array([0.83838384, 0.80808081, 0.7979798 ])